# This is an updated version of the code I uploaded [here](https://www.kaggle.com/code/kaito510/goto-conversion-winning-solution). 
# In other words, this is the 2025 version of the 2024 version I uploaded [here](https://www.kaggle.com/code/kaito510/updated-1xgold-2xsilvers-key-ingredient) to fit the contest design for this year.

The probability matrices were computed by converting betting odds to outcome probabilities using **[goto_conversion](https://github.com/gotoConversion/goto_conversion)**, which are displayed interactively under the first code chunk and can be found [here](https://github.com/gotoConversion/goto_conversion/tree/main/probabilityMatrices) as csv files. I only updated the essential components of the code I uploaded [here](https://www.kaggle.com/code/kaito510/goto-conversion-winning-solution) to ensure I meet the tight deadline.

**In 2024, this solution alone was sufficient for a medal**. This can be verified by noticing [2024's leaderboard scores from 86th to 100th](https://www.kaggle.com/competitions/march-machine-learning-mania-2024/leaderboard) and the leaderboard score of [2024's version of this solution](https://www.kaggle.com/code/kaito510/updated-1xgold-2xsilvers-key-ingredient) are both 0.06035.

For even better performance, this solution should be used as an ingredient for your solution instead of as your entire solution. In 2024, at least **two gold ([3rd](https://www.kaggle.com/competitions/march-machine-learning-mania-2024/discussion/495101) and [4th](https://www.kaggle.com/competitions/march-machine-learning-mania-2024/discussion/494407) place) and one silver ([38th](https://www.kaggle.com/competitions/march-machine-learning-mania-2024/discussion/485888#2740879) place)** medal winners publicly stated that they used this solution as an ingredient for their success; listed [here](https://github.com/gotoConversion/goto_conversion?tab=readme-ov-file#goto_conversion---used-by-4-gold-medal-solutions-on-kaggle).

In [1]:
#Setup

import pandas as pd
year = 2025
kaggleFolderPath = '/kaggle/input/march-machine-learning-mania-' + str(year)
fivethirtyeightFolderPath = '/kaggle/input/538data'

In [2]:
#Mens Probability Matrix
#source: https://github.com/gotoConversion/goto_conversion
#Matrices were computed by converting betting odds to probabilities using goto_conversion

mensProbabilities_df = pd.read_csv(fivethirtyeightFolderPath + '/mensProbabilitiesTable.csv', index_col = 'player') #source: https://github.com/gotoConversion/goto_conversion
mensProbabilities_df = mensProbabilities_df.drop('Elo_Rating', axis=1)

In [3]:
#Womens Probability Matrix
#source: https://github.com/gotoConversion/goto_conversion
#Matrices were computed by converting betting odds to probabilities using goto_conversion

womensProbabilities_df = pd.read_csv(fivethirtyeightFolderPath + '/womensProbabilitiesTable.csv', index_col = 'player') #source: https://github.com/gotoConversion/goto_conversion
womensProbabilities_df = womensProbabilities_df.drop('Elo_Rating', axis=1)

# Submission with Optimal Strategy

**Below is a mathematical proof that the optimal strategy to win a medal under Brier Score is when we assume a team with 33.3% chance of winning a match to win that match.**

The expected return when we risk on a given game can be expressed as:

f(p) = p(1 - p)^2 where p is the probability of success and (1-p)^2 is essentially the reward for the risk taken if the risk succeeds

This implies f'(p) and f''(p) can be expressed as:

f'(p) = -2p + 2p^2 + (1-p)^2

f''(p) = -4 + 6p

argmax_p f(p) = 1/3 with tedious mathematical working omitted.

Thus, expected reward is maximised when we assume a team with 1/3 chance of winning a match to win that match.

In [4]:
#Import team seeds
mensTeamSeeds_df = pd.read_csv(kaggleFolderPath + '/MNCAATourneySeeds.csv')
mensTeamSeeds2025_df = mensTeamSeeds_df.iloc[[x == year for x in mensTeamSeeds_df['Season']]]
womensTeamSeeds_df = pd.read_csv(kaggleFolderPath + '/WNCAATourneySeeds.csv')
womensTeamSeeds2025_df = womensTeamSeeds_df.iloc[[x == year for x in womensTeamSeeds_df['Season']]]

In [5]:
#Implement Optimal Strategy (if you agree)
def get_roundOfMatch(team1, team2, seeds_df):

    slotMap = [1, 16, 8, 9, 5, 12, 4, 13, 6, 11, 3, 14, 7, 10, 2, 15]

    team1_seed = seeds_df.loc[[x == team1 for x in seeds_df['TeamID']],'Seed'].values[0]
    team2_seed = seeds_df.loc[[x == team2 for x in seeds_df['TeamID']],'Seed'].values[0]

    isFirstFourMatch = team1_seed[:3] == team2_seed[:3]
    if isFirstFourMatch:
        return 1

    team1_region = str(team1_seed[:1])
    team2_region = str(team2_seed[:1])

    team1_seedNumber = int(team1_seed[1:3]) #careful with first four teams
    team2_seedNumber = int(team2_seed[1:3]) #careful with first four teams

    isRegionSame = team1_region == team2_region
    if not isRegionSame:

        isTeam1_regionWX = team1_region in ['W','X']
        isTeam2_regionWX = team2_region in ['W','X']

        if isTeam1_regionWX and isTeam2_regionWX: #both W or X region
            return 6

        elif (not isTeam1_regionWX) and (not isTeam2_regionWX): #both not W or X region
            return 6

        else:
            return 7

    else: #same region

        team1_slot = slotMap.index(team1_seedNumber)
        team2_slot = slotMap.index(team2_seedNumber)

        isRound2 = (team1_slot // 2) == (team2_slot // 2)  #round of 64 or first four (not counted anyway)
        if isRound2:
            return 2

        isRound3 = (team1_slot // 4) == (team2_slot // 4)
        if isRound3: #yet to find why but "elif" throws error
            return 3

        isRound4 = (team1_slot // 8) == (team2_slot // 8)
        if isRound4: #yet to find why but "elif" throws error
            return 4

        else:
            return 5

def get_tourneyFlag(team1, team2, seeds_df):

    tourneyTeams = seeds_df['TeamID'].tolist()

    isTeam1InTourney = team1 in tourneyTeams
    isTeam2InTourney = team2 in tourneyTeams

    if isTeam1InTourney and isTeam2InTourney:
        return get_roundOfMatch(team1, team2, seeds_df)

    else:
        return 0

def get_flag_list(submission_df, mensTeamSeeds2025_df, womensTeamSeeds2025_df):
    flag_list = []
    for i in range(submission_df.shape[0]):

        currRow = submission_df.iloc[i,0].split('_')
        team1 = int(currRow[1])
        team2 = int(currRow[2])

        isWomensMatch = team1 + team2 > 6000
        if isWomensMatch:
            flag = get_tourneyFlag(team1, team2, womensTeamSeeds2025_df)
        else:
            flag = get_tourneyFlag(team1, team2, mensTeamSeeds2025_df)

        flag_list.append(flag)
    return flag_list

def set_optimalStrategy(submission_df, mensTeamSeeds2025_df, womensTeamSeeds2025_df, riskTeam, riskTeamToWinRound):

    flag_list = get_flag_list(submission_df, mensTeamSeeds2025_df, womensTeamSeeds2025_df)

    for i in range(submission_df.shape[0]):
        submission_row = submission_df.iloc[i,0].split('_')
        submission_round = flag_list[i]

        team1 = int(submission_row[1])
        team2 = int(submission_row[2])

        isTeam1Win = (team1 == riskTeam) and (0 < submission_round) and (submission_round <= riskTeamToWinRound)
        isTeam2Win = (team2 == riskTeam) and (0 < submission_round) and (submission_round <= riskTeamToWinRound)
        if isTeam1Win:
            submission_df.at[i, 'Pred'] = 1.0
            print(submission_df.iloc[i])
        elif isTeam2Win:
            submission_df.at[i, 'Pred'] = 0.0
            print(submission_df.iloc[i])
    
    return submission_df

submission_df = pd.read_csv(fivethirtyeightFolderPath + '/submission.csv')
riskTeam = 1179 #Drake
riskTeamToWinRound = 2 #Near Optimal Probability for Strategy
submission_df = set_optimalStrategy(submission_df, mensTeamSeeds2025_df, womensTeamSeeds2025_df, riskTeam, riskTeamToWinRound)
submission_df.to_csv('submission.csv', index=False)

ID      2025_1179_1281
Pred               1.0
Name: 23679, dtype: object


In [6]:
updated_df = pd.read_csv('submission.csv')
print(updated_df.head())

               ID  Pred
0  2025_1101_1102   0.5
1  2025_1101_1103   0.5
2  2025_1101_1104   0.5
3  2025_1101_1105   0.5
4  2025_1101_1106   0.5


In [7]:
non_default_preds = updated_df[updated_df['Pred'] != 0.5]
print("Rows where Pred is not 0.5:")
print(non_default_preds)
print("\nTotal count:", len(non_default_preds))

Rows where Pred is not 0.5:
                    ID      Pred
725     2025_1103_1104  0.087146
727     2025_1103_1106  0.519163
730     2025_1103_1110  0.438101
732     2025_1103_1112  0.127970
736     2025_1103_1116  0.397812
...                ...       ...
131004  2025_3452_3456  0.839135
131019  2025_3452_3471  0.852352
131031  2025_3453_3456  0.377695
131046  2025_3453_3471  0.395710
131121  2025_3456_3471  0.519097

[4556 rows x 2 columns]

Total count: 4556


In [8]:
man_champion = 1196
woman_champion = 3376

new_updated_df = updated_df.copy()
for i in range(len(new_updated_df)):
    id_index = new_updated_df.index[i]
    id_str = new_updated_df.at[id_index, 'ID']
    
    year, team1, team2 = map(int, id_str.split('_'))

    # print(f"id_str: {id_str} year: {year} team1: {team1} team2: {team2}")
    
    if man_champion == team1 or woman_champion == team1:
        new_updated_df.at[id_index, 'Pred'] = 1
    elif man_champion == team2 or woman_champion == team2:
        new_updated_df.at[id_index, 'Pred'] = 0

In [9]:
print(len(updated_df))
print(len(new_updated_df))
new_updated_df.head()

131407
131407


,ID,Pred
0,2025_1101_1102,0.5
1,2025_1101_1103,0.5
2,2025_1101_1104,0.5
3,2025_1101_1105,0.5
4,2025_1101_1106,0.5


In [10]:
new_updated_df.to_csv("submission_1.csv", index=False)